# MSigDB ORA Analysis: pathway coverage across sample sizes

**Environment:** `clamp-analyses`

For each CLAMP model (CLAMPfull and CLAMPbase) across all coverage levels (1%, 5%, 10%, 25%, 50%, 75%, 100%) and 3 seeds, this notebook:

1. Loads the Z matrix (gene loadings per LV) for a given coverage level/seed.
2. For each LV, selects the top 1% genes by descending loading.
3. Runs `enricher()` per LV using MSigDB (v2026.1) as gene set database and model genes as universe.
4. Stores raw `terms_padj`: the minimum p.adjust per MSigDB term across all LVs (no FDR threshold applied here).
5. Saves per-seed RDS caches (`rs{pct}_seed{seed}_msigdb.rds`) and per-pct-level summary RDS/CSV. FDR thresholds (0.05 / 0.01) and coverage computation are done in `01_bp_coverage_plot.ipynb`.

`pvalueCutoff = 0.05` (not `1`) for the same reason as the study-coverage/saturation MSigDB notebooks: with no filtering, `enricher()` returns every one of ~35k MSigDB terms per LV (with a verbose `geneID` column), which OOM-killed runs at large K. `0.05` is the loosest FDR threshold used downstream and is behavior-preserving for the coverage calculation.

In [ ]:
library(here)
library(dplyr)
library(clusterProfiler)
library(BiocParallel)

## Paths

In [ ]:
models_dir <- here("output/01_model_building/04_archs4/06_bp_coverage_rshall")
output_dir <- here("output/03_model_biology/00_archs4/05_coverage_random/00_bp_ora_analysis")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(output_dir, "CLAMPbase"),  recursive = TRUE, showWarnings = FALSE)

## Coverage level specs

In [ ]:
coverage_specs <- list(
  list(pct = 1,   dir = "00_bp_coverage_hall_rs_01"),
  list(pct = 5,   dir = "01_bp_coverage_hall_rs_05"),
  list(pct = 10,  dir = "02_bp_coverage_hall_rs_10"),
  list(pct = 25,  dir = "03_bp_coverage_hall_rs_25"),
  list(pct = 50,  dir = "04_bp_coverage_hall_rs_50"),
  list(pct = 75,  dir = "05_bp_coverage_hall_rs_75"),
  list(pct = 100, dir = "06_bp_coverage_hall_rs_100")
)

seeds <- 1:3

## Load MSigDB gene sets

In [ ]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
message(sprintf("MSigDB gene sets loaded: %d", length(unique(msig_gmt$term))))

## Helper: run ORA for one model (one seed, one coverage level)

Returns a list with raw `terms_padj` (minimum p.adjust per MSigDB term across all LVs).

In [ ]:
run_ora_for_model <- function(z_path, n_cores = 4) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  n_top          <- ceiling(0.01 * nrow(Z))
  n_lvs          <- ncol(Z)

  term_overlap   <- tapply(msig_gmt$gene %in% universe_genes, msig_gmt$term, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  top_genes_per_lv <- apply(Z, 2, function(lv) {
    universe_genes[order(lv, decreasing = TRUE)[seq_len(n_top)]]
  })

  # n_samples via B.csv header only (avoids loading the full multi-GB model rds)
  b_path <- file.path(dirname(z_path), "B.csv")
  n_samples <- if (file.exists(b_path)) {
    ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
  } else NA_integer_

  bp <- BiocParallel::MulticoreParam(workers = n_cores, progressbar = FALSE)
  ora_results <- BiocParallel::bplapply(
    seq_len(n_lvs),
    function(i) {
      genes <- top_genes_per_lv[, i]
      tryCatch(
        clusterProfiler::enricher(
          gene          = genes,
          universe      = universe_genes,
          TERM2GENE     = msig_gmt,
          pAdjustMethod = "BH",
          pvalueCutoff  = 0.05,
          qvalueCutoff  = 1,
          minGSSize     = 10,
          maxGSSize     = 50000
        ),
        error = function(e) NULL
      )
    },
    BPPARAM = bp
  )

  all_dfs <- lapply(ora_results, function(r) {
    if (is.null(r) || nrow(as.data.frame(r)) == 0) return(NULL)
    as.data.frame(r)
  })
  all_dfs <- Filter(Negate(is.null), all_dfs)

  if (length(all_dfs) == 0) {
    warning("No ORA results returned for: ", z_path)
    return(NULL)
  }

  combined <- do.call(rbind, all_dfs)

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_top_genes    = n_top,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(combined$p.adjust, combined$ID, min)
  )
}

## Run ORA: CLAMPfull

In [ ]:
results_clampfull_by_pct <- list()

for (spec in coverage_specs) {
  pct_rds_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%d_msigdb.rds", spec$pct))

  if (file.exists(pct_rds_path)) {
    message(sprintf("Loading cached pct-level result: CLAMPfull %d%%", spec$pct))
    results_clampfull_by_pct[[as.character(spec$pct)]] <- readRDS(pct_rds_path)
    next
  }

  pct_rows <- lapply(seeds, function(s) {
    seed_dir   <- file.path(models_dir, spec$dir, sprintf("hall_coverage_rs%d_seed_%d", spec$pct, s))
    z_path     <- file.path(seed_dir, "CLAMPfull_hall", "Z.csv")
    cache_path <- file.path(output_dir, "CLAMPfull", sprintf("rs%d_seed%d_msigdb.rds", spec$pct, s))

    if (!file.exists(z_path)) { warning("Z.csv not found: ", z_path); return(NULL) }

    if (file.exists(cache_path)) {
      message(sprintf("Loading cached: CLAMPfull rs%d seed%d", spec$pct, s))
      res <- readRDS(cache_path)
    } else {
      message(sprintf("Running ORA: CLAMPfull rs%d seed%d", spec$pct, s))
      res <- run_ora_for_model(z_path)
      if (!is.null(res)) saveRDS(res, cache_path)
    }
    if (is.null(res)) return(NULL)

    data.frame(
      model_type     = "CLAMPfull",
      coverage_pct   = spec$pct,
      seed           = s,
      n_samples      = res$n_samples,
      n_lvs          = res$n_lvs,
      n_top_genes    = res$n_top_genes,
      n_total_msigdb = res$n_total_msigdb,
      stringsAsFactors = FALSE
    )
  })

  pct_df <- do.call(rbind, Filter(Negate(is.null), pct_rows))
  rownames(pct_df) <- NULL
  results_clampfull_by_pct[[as.character(spec$pct)]] <- pct_df
  saveRDS(pct_df, pct_rds_path)
  message(sprintf("Saved: CLAMPfull %d%% -> %s", spec$pct, pct_rds_path))
}

results_clampfull_df <- do.call(rbind, results_clampfull_by_pct)
rownames(results_clampfull_df) <- NULL
print(results_clampfull_df)

In [ ]:
for (pct in names(results_clampfull_by_pct)) {
  csv_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%s_msigdb.csv", pct))
  write.csv(results_clampfull_by_pct[[pct]], csv_path, row.names = FALSE)
  message("Saved: ", csv_path)
}

## Run ORA: CLAMPbase

In [ ]:
results_clampbase_by_pct <- list()

for (spec in coverage_specs) {
  pct_rds_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%d_msigdb.rds", spec$pct))

  if (file.exists(pct_rds_path)) {
    message(sprintf("Loading cached pct-level result: CLAMPbase %d%%", spec$pct))
    results_clampbase_by_pct[[as.character(spec$pct)]] <- readRDS(pct_rds_path)
    next
  }

  pct_rows <- lapply(seeds, function(s) {
    seed_dir   <- file.path(models_dir, spec$dir, sprintf("hall_coverage_rs%d_seed_%d", spec$pct, s))
    z_path     <- file.path(seed_dir, "CLAMPbase", "Z.csv")
    cache_path <- file.path(output_dir, "CLAMPbase", sprintf("rs%d_seed%d_msigdb.rds", spec$pct, s))

    if (!file.exists(z_path)) { warning("Z.csv not found: ", z_path); return(NULL) }

    if (file.exists(cache_path)) {
      message(sprintf("Loading cached: CLAMPbase rs%d seed%d", spec$pct, s))
      res <- readRDS(cache_path)
    } else {
      message(sprintf("Running ORA: CLAMPbase rs%d seed%d", spec$pct, s))
      res <- run_ora_for_model(z_path)
      if (!is.null(res)) saveRDS(res, cache_path)
    }
    if (is.null(res)) return(NULL)

    data.frame(
      model_type     = "CLAMPbase",
      coverage_pct   = spec$pct,
      seed           = s,
      n_samples      = res$n_samples,
      n_lvs          = res$n_lvs,
      n_top_genes    = res$n_top_genes,
      n_total_msigdb = res$n_total_msigdb,
      stringsAsFactors = FALSE
    )
  })

  pct_df <- do.call(rbind, Filter(Negate(is.null), pct_rows))
  rownames(pct_df) <- NULL
  results_clampbase_by_pct[[as.character(spec$pct)]] <- pct_df
  saveRDS(pct_df, pct_rds_path)
  message(sprintf("Saved: CLAMPbase %d%% -> %s", spec$pct, pct_rds_path))
}

results_clampbase_df <- do.call(rbind, results_clampbase_by_pct)
rownames(results_clampbase_df) <- NULL
print(results_clampbase_df)

In [ ]:
for (pct in names(results_clampbase_by_pct)) {
  csv_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%s_msigdb.csv", pct))
  write.csv(results_clampbase_by_pct[[pct]], csv_path, row.names = FALSE)
  message("Saved: ", csv_path)
}